Report Capstone Project

Tobias van der Meer, Rafael Lima Araujo Rego, Janniek Graveland, Jorrit Paques and Bram Verlaat

This notebook is not runable as it shows only parts of our code that shows the model's architecture. The whole code is sent in an other file.

In this report, we will briefly explain the architectures of both the CNN model and UNet model. Next to giving an explanation, we will also show the code of the architecture of our models. The two best models are explained in detail and all the codes of the other model architectures are briefly explained as well. This notebook is not runable as it shows only parts of our code that shows the model's architecture. The whole code is sent in an other file.

Datasets and splits:
The datasets are generated by our supervisor Jeffrey using the code below, which relies on functions provided by our supervisor. Each sample consists of a hydraulic conductivity field k(x,y) and the corresponding hydraulic head field h(x,y).

In [1]:
from jeffrey_code import hydraulic_conductivity_field, solve_darcy_flow, source_function
import numpy as np

n = 64

def generate_data(seeds):
    x = np.empty((len(seeds), n**2))
    y = np.empty((len(seeds), n**2))

    for i, seed in enumerate(seeds):
        print(i)
        x[i, :] = hydraulic_conductivity_field(n, seed)
        f = source_function(n)
        y[i, :] = solve_darcy_flow(n, x[i, :], f)
    return x, y

def save_data(x, y, id=""):
    np.savetxt(f"datasets/k_set{id}.txt", x.reshape(-1, n*n))
    np.savetxt(f"datasets/h_set{id}.txt", y.reshape(-1, n*n))

if __name__ == "__main__":
    start_seed, stop_seed = 0, 1000
    print(f"computing seeds {start_seed} to {stop_seed}")
    x, y =generate_data(list(range(start_seed, stop_seed)))
    save_data(x, y, id=f"_{start_seed}to{stop_seed}")

ModuleNotFoundError: No module named 'jeffrey_code'

Initially, all data were generated on a 60x60 grid. Subsequently, a 64x64 grid was adopted, as it is divisible by powers of two and therefore better suited for deeper CNN and U-Net architectures. The 60x60 grid is used for most model experiments, while the 64x64 grid is used for the best-performing CNN and U-Net models. Our assumption is that the general performance of less deep models wont significantly change when implementing the 64x64 grid, as the grids hydraulic conductivity fields are computed with the same seed and therefore result in similar data. So, for computational time, we realised it is too late, we only used the 64x64 grid on our best models.

Each dataset is stored as(regarding the 60x60 grid):
• datasets/k set AtoB.txt: each row contains a flattened conductivity field k (length 3600 for
60 × 60).
• datasets/h set AtoB.txt: each row contains the corresponding flattened head field h.
The suffix AtoB denotes the random seed range used during data generation.
A consistent train/test split is used across all CNN and U-Net models:
• Training set: seed ranges [0, 1400–2000, 2000–3000, . . . , 7000–8000]
• Test set: seed ranges [1000–1050, 1050–1400]
This results in 7,600 training samples and 400 test samples.
Final model evaluation is performed using a separate validation dataset, generated independently from
the training and test sets.

CNN-models and baseline model:

CNN fc1 - At last, this is our baseline CNN, which is a fully connected model, without convolutional layers. The results are suprisingly smooth, which means that convolutional layers are not necessary needed to get very smooth results. The MAE os 2.50

In [ ]:
class Model(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(3600, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 512)
        self.fc4 = nn.Linear(512, 512)
        self.fc5 = nn.Linear(512, 3600)

    def forward(self, x):
        x = x.reshape((-1, 3600))
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.fc5(x)
        return x.reshape((-1, 60, 60))

CNN12: This model uses four blocks and a double fully connected layes. The idea is that the first 2 fully connected layers compute a estimate of the h we want to predict, and the blocks iteratively compute a refinement to this estimate. Each block has four convolutional layers. The input to these convolutional layers consists of four channels, one for the raw input (the conductivity), one for the most recent estimate of h, and the other two channels use the same date but first apply a double fully connected layer to help generate the global structure of the h-field. We used double fully connected layers instead of single ones because this reduces the number of parameters because we ues a small number of hidden layers.

In [ ]:
class Block2(nn.Module):
    def __init__(self, n_hidden=144):
        super().__init__()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(3600, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 3600)
        self.fc3 = nn.Linear(3600, n_hidden)
        self.fc4 = nn.Linear(n_hidden, 3600)
        self.prep1 = nn.Sequential(self.fc1, self.relu, self.fc2, self.relu)
        self.prep2 = nn.Sequential(self.fc3, self.relu, self.fc4, self.relu)
        self.conv1 = nn.Conv2d(4, 8, 9, padding='same', padding_mode='zeros')
        self.conv2 = nn.Conv2d(8, 8, 9, padding='same', padding_mode='zeros')
        self.conv3 = nn.Conv2d(8, 8, 9, padding='same', padding_mode='zeros')
        self.conv4 = nn.Conv2d(8, 1, 9, padding='same', padding_mode='zeros')

    def forward(self, x, hr):
        z = torch.empty((x.shape[0], 4, 60, 60), device=x.device)
        z[:, 0, :, :] = x.view(-1, 60, 60)
        z[:, 1, :, :] = self.prep1(x.view(-1, 3600)).view(-1, 60, 60)
        z[:, 2, :, :] = hr.view(-1, 60, 60)
        z[:, 3, :, :] = self.prep2(hr.view(-1, 3600)).view(-1, 60, 60)

        r = self.relu(self.conv1(z))
        r = self.relu(self.conv2(r))
        r = self.relu(self.conv3(r))
        r = self.conv4(r)
        return r

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(3600, 32)
        self.fc2 = nn.Linear(32, 3600)
        self.block_1 = Block2(n_hidden=32)
        self.block_2 = Block2(n_hidden=64)
        self.block_3 = Block2(n_hidden=64)
        self.block_4 = Block2(n_hidden=128)


    def forward(self, x):
        h = self.relu(self.fc1(x.reshape((-1, 3600))))
        h = self.relu(self.fc2(h))
        h = h.reshape((-1, 1, 60, 60))
        h = h - self.block_1(x, h)
        h = h - self.block_2(x, h)
        h = h - self.block_3(x, h)
        h = h - self.block_4(x, h)
        return h.reshape((-1, 60, 60))


CNN12c - In this model, this is a modified version of cnn12, which where i placed more hidden layers at the start and less at the end, keeping the
number of parameters similar. The ideas is that this would reduce the noise in the results.

In [ ]:
class Block2(nn.Module):
    def __init__(self, n_hidden=144):
        super().__init__()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(3600, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 3600)
        self.fc3 = nn.Linear(3600, n_hidden)
        self.fc4 = nn.Linear(n_hidden, 3600)
        self.prep1 = nn.Sequential(self.fc1, self.relu, self.fc2, self.relu)
        self.prep2 = nn.Sequential(self.fc3, self.relu, self.fc4, self.relu)
        self.conv1 = nn.Conv2d(4, 8, 9, padding='same', padding_mode='zeros')
        self.conv2 = nn.Conv2d(8, 8, 9, padding='same', padding_mode='zeros')
        self.conv3 = nn.Conv2d(8, 8, 9, padding='same', padding_mode='zeros')
        self.conv4 = nn.Conv2d(8, 1, 9, padding='same', padding_mode='zeros')

    def forward(self, x, hr):
        z = torch.empty((x.shape[0], 4, 60, 60), device=x.device)
        z[:, 0, :, :] = x.view(-1, 60, 60)
        z[:, 1, :, :] = self.prep1(x.view(-1, 3600)).view(-1, 60, 60)
        z[:, 2, :, :] = hr.view(-1, 60, 60)
        z[:, 3, :, :] = self.prep2(hr.view(-1, 3600)).view(-1, 60, 60)

        r = self.relu(self.conv1(z))
        r = self.relu(self.conv2(r))
        r = self.relu(self.conv3(r))
        r = self.conv4(r)
        return r

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(3600, 128)
        self.fc2 = nn.Linear(128, 3600)
        self.block_1 = Block2(n_hidden=128)
        self.block_2 = Block2(n_hidden=64)
        self.block_3 = Block2(n_hidden=32)
        self.block_4 = Block2(n_hidden=32)


    def forward(self, x):
        h = self.relu(self.fc1(x.reshape((-1, 3600))))
        h = self.relu(self.fc2(h))
        h = h.reshape((-1, 1, 60, 60))
        h = h - self.block_1(x, h)
        h = h - self.block_2(x, h)
        h = h - self.block_3(x, h)
        h = h - self.block_4(x, h)
        return h.reshape((-1, 60, 60))

CNN 16 - In this model has 12 convolutional layers together with three residual connections

In [ ]:
class Model(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.conv1 = nn.Conv2d(1, 8, 11, padding='same', padding_mode='zeros')
        self.conv2 = nn.Conv2d(8, 16, 11, padding='same', padding_mode='zeros')

        self.conv3 = nn.Conv2d(16, 16, 19, padding='same', padding_mode='zeros')
        self.conv4 = nn.Conv2d(16, 16, 19, padding='same', padding_mode='reflect')
        self.conv5 = nn.Conv2d(16, 16, 19, padding='same', padding_mode='reflect')

        self.conv6 = nn.Conv2d(16, 16, 13, padding='same', padding_mode='zeros')
        self.conv7 = nn.Conv2d(16, 16, 13, padding='same', padding_mode='reflect')
        self.conv8 = nn.Conv2d(16, 16, 13, padding='same', padding_mode='reflect')

        self.conv9 = nn.Conv2d(16, 16, 9, padding='same', padding_mode='zeros')
        self.conv10 = nn.Conv2d(16, 16, 9, padding='same', padding_mode='zeros')
        self.conv11 = nn.Conv2d(16, 16, 9, padding='same', padding_mode='zeros')

        self.conv12 = nn.Conv2d(16, 1, 7, padding='same', padding_mode='reflect')

    def forward(self, x):
        h1 = self.relu(self.conv1(x))
        h1 = self.relu(self.conv2(h1))

        h2 = self.relu(self.conv3(h1))
        h2 = self.relu(self.conv4(h2))
        h2 = self.relu(self.conv5(h2)) + h1

        h3 = self.relu(self.conv6(h2))
        h3 = self.relu(self.conv7(h3))
        h3 = self.relu(self.conv8(h3)) + h2

        h4 = self.relu(self.conv9(h3))
        h4 = self.relu(self.conv10(h4))
        h4 = self.relu(self.conv11(h4)) + h3

        h5 = self.conv12(h4)
        return h5.view(-1, 60, 60)

The best CNN-model: 
The best performing convolutionial neural network model is the one that starts with a convolutional part, which is very similar to the encoder part of a u-net, and ends with three fully connected layers. It takes a single-channel 2D-input and outputs a 60x60 2D field. It It gives a MAE of 1.053 when trained, and it generates a smooth output field which we will show below. 

In [ ]:
class Model(nn.Module):
    # During one of the meetings the idea of putting a fully connected layer behind a convolutional one came up.
    # This model starts with a convolutional part that is very similar to the encoder part of a u-net, after that
    # it has three fully connected layers. This model works very well, giving a MEA of 1.053 when trained. The model
    # also generates a relatively smooth output field and it does not have bad outliers

    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.conv1 = nn.Conv2d(1, 16, 5, padding='same', padding_mode='reflect')
        self.conv2 = nn.Conv2d(16, 16, 7, padding='same', padding_mode='zeros')
        self.pool1 = nn.MaxPool2d(2)  # 30x30
        self.conv3 = nn.Conv2d(16, 32, 5, padding=3, padding_mode='zeros')  # 32x32
        self.conv4 = nn.Conv2d(32, 32, 5, padding='same', padding_mode='zeros')
        self.pool2 = nn.MaxPool2d(2)  # 16x16
        self.conv5 = nn.Conv2d(32, 64, 5, padding='same', padding_mode='zeros')
        self.conv6 = nn.Conv2d(64, 64, 5, padding='same', padding_mode='zeros')
        self.pool3 = nn.MaxPool2d(2) # 8x8
        self.conv7 = nn.Conv2d(64, 64, 5, padding='same', padding_mode='zeros')
        self.conv8 = nn.Conv2d(64, 64, 3, padding='same', padding_mode='zeros')

        self.fc1 = nn.Linear(4096, 2048)
        self.fc2 = nn.Linear(2048, 2048)
        self.fc3 = nn.Linear(2048, 3600)



    def forward(self, x):
        #convolutional part
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        h = self.pool1(h)

        h = self.relu(self.conv3(h))
        h = self.relu(self.conv4(h))
        h = self.pool2(h)

        h = self.relu(self.conv5(h))
        h = self.relu(self.conv6(h))
        h = self.pool3(h)

        h = self.relu(self.conv7(h))
        h = self.relu(self.conv8(h)).view(-1, 4096)

        # fully connected part
        h = self.relu(self.fc1(h))
        h = self.relu(self.fc2(h))
        h = self.fc3(h)

        return h.reshape((-1, 60, 60))

Unet-models:

The best Unet-model:
The best Unet-model is specifically for a 64x64 grid for which it is possible. It is built up with an encoder-bottleneck-decoder structure. It encodes local features and compresses this to the bottleneck. The key part of this model architecture is the so-called "global brain". Where a standard Unet often fails on global linear gradients, this model takes into account these global features.  

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x


class Model(nn.Module):
    """
    Deeper U-Net for 64x64 grid.
    Depth: 3 pools => bottleneck at 8x8
    Includes global injection in bottleneck.
    """

    def __init__(self, base_ch: int = 64, enforce_dirichlet_row0: bool = False):
        super().__init__()
        self.enforce_dirichlet_row0 = enforce_dirichlet_row0

        in_ch = 1

        # ================= Encoder =================
        # 64x64
        self.enc1 = ConvBlock(in_ch, base_ch)
        self.pool1 = nn.MaxPool2d(2)  # 64 -> 32

        # 32x32
        self.enc2 = ConvBlock(base_ch, 2 * base_ch)
        self.pool2 = nn.MaxPool2d(2)  # 32 -> 16

        # 16x16
        self.enc3 = ConvBlock(2 * base_ch, 4 * base_ch)
        self.pool3 = nn.MaxPool2d(2)  # 16 -> 8

        # Bottleneck 8x8
        self.center = ConvBlock(4 * base_ch, 8 * base_ch)

        # ================= Global Injection =================
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        feature_ch = 8 * base_ch

        self.global_dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_ch, feature_ch),
            nn.ReLU(inplace=True),
            nn.Linear(feature_ch, feature_ch),
            nn.Unflatten(1, (feature_ch, 1, 1))
        )

        # ================= Decoder =================
        # 8 -> 16
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec3 = ConvBlock(8 * base_ch + 4 * base_ch, 4 * base_ch)

        # 16 -> 32
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = ConvBlock(4 * base_ch + 2 * base_ch, 2 * base_ch)

        # 32 -> 64
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = ConvBlock(2 * base_ch + base_ch, base_ch)

        self.out = nn.Conv2d(base_ch, 1, kernel_size=1)

        self.dirichlet_row0_value = (100.0 - 145.3243) / 35.5957

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, 1, 64, 64)

        # ----- Encoder -----
        x1 = self.enc1(x)                 # 64x64
        x2 = self.enc2(self.pool1(x1))    # 32x32
        x3 = self.enc3(self.pool2(x2))    # 16x16

        # ----- Bottleneck -----
        x_center = self.pool3(x3)         # 8x8
        x_center = self.center(x_center)

        # Global Injection
        global_feat = self.global_pool(x_center)
        global_feat = self.global_dense(global_feat)
        x_center = x_center + global_feat

        # ----- Decoder -----
        d3 = self.up3(x_center)           # 16x16
        d3 = torch.cat([d3, x3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)                 # 32x32
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)                 # 64x64
        d1 = torch.cat([d1, x1], dim=1)
        d1 = self.dec1(d1)

        out = self.out(d1)

        if self.enforce_dirichlet_row0:
            out[:, :, 0, :] = self.dirichlet_row0_value

        return out